# Apply Author ORCID Curations

Detects which authors' displayed ORCID will change under current ORCID curations and enqueues their `work_ids` for ES re-sync. ORCID rides in the author `ids` struct on works, so a corrected/detached ORCID must refresh those works in ES/API. Mirrors `ApplyAuthorNameCurations`.

The ORCID itself is applied as an **override** in `CreateAuthors` (replace / detach / organic), not written in-place to `openalex.authors.authors`. Deleting a curation automatically reverts the author to their organic ORCID on the next `CreateAuthors` run.

Runs in `jobs/authors.yaml` after `SyncAuthorOrcidCurations` and before `CreateAuthors`.

```
SyncAuthorOrcidCurations
  ↓
ApplyAuthorOrcidCurations   ← this notebook (diff + enqueue)
  ↓
CreateAuthors               ← applies replace/detach/organic override
  ↓ openalex_authors
  ↓ next end2end cycle
UpdateWorkAuthorships re-emits works → sync_works → ES → API
```

## Diff logic

Compare each author's **currently displayed** ORCID (`openalex_authors.orcid`) against what it **should be** under current curations — the same `replace`/`detach`/`organic` CASE used in `CreateAuthors`. One comparison catches all cases:
- **New replace**: displayed = organic, should = curated → differs → enqueue
- **Changed replace**: displayed = old curated, should = new curated → differs → enqueue
- **New remove**: displayed = wrong organic, should = NULL → differs → enqueue
- **Deleted curation**: displayed = curated (or NULL), should = organic → differs → enqueue

## Stage diff (which authors' displayed ORCID will change)

In [ ]:
%sql
-- Scoped to authors who either (a) have an active ORCID curation or (b) have a displayed ORCID
-- that already differs from their organic ORCID (a previously-applied or now-deleted curation).
-- The second condition catches curation deletions.
CREATE OR REPLACE TABLE openalex.authors.author_orcid_pending_changes AS
SELECT a.id AS author_id
FROM openalex.authors.authors a
JOIN openalex.authors.openalex_authors oa ON a.id = oa.id
LEFT JOIN openalex.authors.author_orcid_curations oc ON a.id = oc.author_id
WHERE (oc.author_id IS NOT NULL OR NOT (oa.orcid <=> a.orcid))
  AND NOT (oa.orcid <=> CASE
    WHEN oc.curated_orcid IS NOT NULL AND oc.curated_orcid <> '' THEN oc.curated_orcid
    WHEN oc.removed_orcid IS NOT NULL AND a.orcid = oc.removed_orcid THEN NULL
    ELSE a.orcid
  END)

## Enqueue affected work_ids for ES re-sync

In [ ]:
%sql
-- Enqueue every work_id where an ORCID-changed author appears in work_authors.
INSERT INTO openalex.works.curated_work_ids_pending_sync
SELECT DISTINCT wa.work_id, current_timestamp() AS added_datetime
FROM openalex.works.work_authors wa
JOIN openalex.authors.author_orcid_pending_changes p ON wa.author_id = p.author_id

## Verify

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM openalex.authors.author_orcid_curations)        AS total_curations,
  (SELECT COUNT(*) FROM openalex.authors.author_orcid_pending_changes)  AS authors_changed,
  (SELECT COUNT(DISTINCT wa.work_id)
     FROM openalex.works.work_authors wa
     JOIN openalex.authors.author_orcid_pending_changes p
       ON wa.author_id = p.author_id)                                   AS works_enqueued

In [ ]:
%sql
-- Spot-check: curated authors showing organic vs curated/removed vs currently-displayed ORCID.
SELECT
  oc.author_id,
  a.orcid           AS organic_orcid,
  oc.curated_orcid,
  oc.removed_orcid,
  oa.orcid          AS currently_displayed
FROM openalex.authors.author_orcid_curations oc
LEFT JOIN openalex.authors.authors a ON a.id = oc.author_id
LEFT JOIN openalex.authors.openalex_authors oa ON oa.id = oc.author_id
ORDER BY oc.updated_datetime DESC NULLS LAST
LIMIT 100